In [1]:
# Source - https://stackoverflow.com/a
# Posted by G M, modified by community. See post 'Timeline' for change history
if 'google.colab' in str(get_ipython()):
  !git clone https://github.com/Vladislavicious/jenga_ml.git
  %cd jenga_ml
  !git switch dev

  !pip install -r requirements.txt
else:
  print('Not running on CoLab')

Not running on CoLab


In [2]:
import random
from environment import make_jenga_env
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv
from stable_baselines3.common.env_util import make_vec_env


In [3]:
n_blocks = 6
random.seed(123)
np.random.seed(123)
CHECK_STEPS = 1024

def make_env():
    return make_jenga_env(n_blocks=n_blocks, render=True)


In [4]:
num_envs = 8
env = make_vec_env(make_env, n_envs=num_envs, vec_env_cls=SubprocVecEnv)


In [ ]:

model = PPO(
    "MlpPolicy",
    env,
    n_steps=CHECK_STEPS,
    batch_size=128,
    verbose=1,
    seed=123
)

model.learn(total_timesteps=200_000)

Using cpu device
-----------------------------
| time/              |      |
|    fps             | 1644 |
|    iterations      | 1    |
|    time_elapsed    | 4    |
|    total_timesteps | 8192 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 1211        |
|    iterations           | 2           |
|    time_elapsed         | 13          |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.017391779 |
|    clip_fraction        | 0.232       |
|    clip_range           | 0.2         |
|    entropy_loss         | -8.37       |
|    explained_variance   | -2.6        |
|    learning_rate        | 0.001       |
|    loss                 | -0.0909     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.0188     |
|    value_loss           | 0.0244      |
-----------------------------------------
-----------------

In [6]:
model.save("jenga_ppo_multithread")


In [7]:
single_env = make_env()

In [13]:
obs, _ = single_env.reset()
for _ in range(CHECK_STEPS):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = single_env.step(action)
    single_env.render()
    if terminated or truncated:
        obs, _ = single_env.reset()
    single_env.env.debug_output()

block 0: [-0.39952321 -0.23983145  0.00718846]
block 1: [ 0.35959672 -0.10557082  0.00737709]
block 2: [-0.1972078  -0.0035685   0.00730627]
block 3: [0.35674952 0.14831785 0.00732996]
block 4: [-0.06026627 -0.12204275  0.00734924]
block 5: [ 0.06237374 -0.27005441  0.00736979]
block 0: [-0.39954442 -0.23981699  0.00724096]
block 1: [ 0.35959582 -0.10556894  0.00738077]
block 2: [-0.19720613 -0.00356848  0.0073274 ]
block 3: [0.36028506 0.14478232 0.00732996]
block 4: [-0.06027164 -0.12203973  0.0073598 ]
block 5: [ 0.06237306 -0.2700543   0.00737525]
block 0: [-0.39955989 -0.23980644  0.00727925]
block 1: [ 0.36459582 -0.10556894  0.00738077]
block 2: [-0.19720487 -0.00356845  0.00734328]
block 3: [0.3602872  0.14479017 0.00734586]
block 4: [-0.06027568 -0.12203745  0.00736774]
block 5: [ 0.06237254 -0.27005422  0.00737937]
block 0: [-0.39957132 -0.23979866  0.0073075 ]
block 1: [ 0.36459581 -0.10556893  0.01519121]
block 2: [-0.19720393 -0.00356844  0.00735524]
block 3: [0.36028883 0

In [9]:
model.save("model_100k.mod")

In [10]:
env.env.debug_output()

AttributeError: 'SubprocVecEnv' object has no attribute 'env'